# पाठ 10 - AI एजेंट्स इन प्रोडक्शन

इस पाठ में आप Microsoft Agent Framework (Python) का उपयोग करके AI एजेंट्स के लिए **प्रोडक्शन पैटर्न** सीखेंगे। हम कवर करते हैं:

- **Observability** — एजेंट इंटरैक्शन में समय मापन और लॉगिंग जोड़ना
- **Evaluation** — प्रतिक्रिया की गुणवत्ता स्कोर करने के लिए एक मूल्यांकन एजेंट का उपयोग
- **Cost management** — टोकन अनुकूलन और मॉडल चयन के लिए रणनीतियाँ

परिदृश्य एक **यात्रा एजेंट** है जो उपयोगकर्ताओं को यात्राओं की योजना बनाने में मदद करता है, जिसके ऊपर निगरानी और मूल्यांकन परतें जोड़ी गई हैं।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## उत्पादन संबंधी विचार

प्रोटोटाइप से उत्पादन तक AI एजेंट्स के स्थानांतरण के लिए तीन स्तंभों पर सावधानीपूर्वक ध्यान देने की आवश्यकता होती है:

1. **निगरानी** — आपको यह देखने की आवश्यकता है कि एजेंट क्या कर रहा है, इसमें कितना समय लग रहा है, और यह किन टूल्स को कॉल करता है। ट्रेसिंग और लॉगिंग के बिना उत्पादन समस्याओं का डिबग लगभग असंभव होता है।

2. **मूल्यांकन** — स्वचालित गुणवत्ता जाँच यह सुनिश्चित करती है कि एजेंट की प्रतिक्रियाएँ समय के साथ सटीक, पूर्ण, और सहायक बनी रहें। एक मूल्यांकन एजेंट परिभाषित मानदंडों के खिलाफ प्रतिक्रियाओं को स्कोर कर सकता है।

3. **लागत प्रबंधन** — टोकन उपयोग सीधे लागत को प्रभावित करता है। प्रॉम्प्ट अनुकूलन, मॉडल चयन, और कैशिंग जैसी रणनीतियाँ बिना गुणवत्ता का त्याग किए खर्चों को नियंत्रण में रखने में मदद करती हैं।


## एक अवलोकनीय एजेंट बनाना

हम यात्रा उपकरणों को परिभाषित करते हैं और लेटेंसी की निगरानी के लिए एजेंट कॉल को समय मापन के साथ लपेटते हैं। उत्पादन में आप OpenTelemetry या इसी तरह के ट्रेसिंग बैकएंड के साथ एकीकृत करेंगे।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन पैटर्न

एक सामान्य प्रोडक्शन पैटर्न यह है कि दूसरे एजेंट का उपयोग एक **मूल्यांकनकर्ता** के रूप में किया जाए। मूल्यांकनकर्ता प्राथमिक एजेंट की प्रतिक्रिया को पूर्वनिर्धारित मानदंडों जैसे पूर्णता, सटीकता, और सहायकता के आधार पर स्कोर करता है।

यह सक्षम बनाता है:
- उपयोगकर्ताओं तक प्रतिक्रियाएँ पहुँचने से पहले स्वचालित गुणवत्ता जाँच
- जब प्रॉम्प्ट या मॉडल बदलते हैं तो रिग्रेशन का पता लगाना
- समय के साथ एजेंट के प्रदर्शन की सतत निगरानी


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत प्रबंधन रणनीतियाँ

उत्पादन AI एजेंट्स के लिए लागतों को नियंत्रित करना अत्यंत महत्वपूर्ण है। यहाँ प्रमुख रणनीतियाँ दी गई हैं:

| Strategy | Description |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | सिस्टम निर्देशों को संक्षिप्त रखें। इनपुट टोकन घटाने के लिए अनावश्यक संदर्भ हटा दें। |
| **मॉडल चयन** | सरल कार्यों जैसे वर्गीकरण या निष्कर्षण के लिए छोटे, सस्ते मॉडल (जैसे GPT-4o-mini) का उपयोग करें, और जटिल तर्क के लिए बड़े मॉडल सुरक्षित रखें। |
| **कैशिंग** | टूल के परिणामों और बार-बार आने वाले प्रश्नों को कैश करें ताकि अनावश्यक API कॉल से बचा जा सके। |
| **टोकन बजट** | अप्रत्याशित रूप से लंबी प्रतिक्रियाएँ रोकने के लिए `max_tokens` सीमाएँ स्थापित करें। |
| **बैचिंग** | जहाँ संभव हो, कई उपयोगकर्ता प्रश्नों को एकल API कॉल में समूहित करें। |

व्यवहार में, एक स्तरित दृष्टिकोण अच्छा काम करता है: सरल अनुरोधों को तेज़, सस्ते मॉडल की ओर निर्देशित करें और केवल जटिल प्रश्नों के लिए अधिक सक्षम मॉडल का उपयोग करें।


## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. **निरीक्षणीयता जोड़ें** एजेंट इंटरैक्शन में समय-निर्धारण और लॉगिंग के साथ, ट्रेसिंग और मॉनिटरिंग के लिए बुनियादी आधार तैयार करते हुए।
2. **एजेंट प्रतिक्रियाओं का मूल्यांकन करें** स्वचालित रूप से एक मूल्यांकनकर्ता एजेंट का उपयोग करके जो पूर्णता, सटीकता, और सहायकता को स्कोर करता है।
3. **लागत प्रबंधन करें** प्रॉम्प्ट अनुकूलन, मॉडल चयन, कैशिंग, और टोकन बजट के माध्यम से।

ये प्रोडक्शन पैटर्न सुनिश्चित करने में मदद करते हैं कि आपके एआई एजेंट बड़े पैमाने पर विश्वसनीय, मापनीय, और लागत-कुशल हों।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या असमानताएँ हो सकती हैं। मूल दस्तावेज़ को उसकी मूल भाषा में प्राधिकृत स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की अनुशंसा की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
